In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%load_ext autoreload
%autoreload 2


from new_model.parser_functions import PlanParser, DocumentParser
from new_model.retriever import Retriever
from new_model.embeddings import get_embeddings
from services.procurement_reference_registry import ProcurementReferenceRegistry

In [2]:
import re
from pathlib import Path
from typing import Any, Dict, Optional, List

from new_model.parser_functions import DocumentParser, PlanParser, parse_okpd_entries, parse_ktry_entries, _clean_keyword_dict, _extract_keyword_windows
from new_model.retriever import Retriever, BM25TextRetriever

from govno_model.docs_parsing import _parse_plan_points, _parse_contract_points, _parse_contract_characteristics, _parse_ooz_points, _parse_zapiska_text, _parse_onmck_text, _parse_onmck_pricies
from govno_model.rag_processing import process_rag_points
from govno_model.smart_processing import process_smart_points 
from govno_model.check_registry import get_regestry_response_okpd_ktry 

contract_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\\doci_primery\Данные для тестирования 01.06.26\4. Проект контракта.docx")
plan_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\doci_primery\Данные для тестирования 01.06.26\1_Заявка_на_включение_в_план_график.docx")
zapiska_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\Данные для тестирования 01.06.26\5. Пояснительная_записка.docx")
OOZ_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\Данные для тестирования 01.06.26\3. ООЗ_Лицензии_на_ТД.docx")
ONMCK_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\Данные для тестирования 01.06.26\2. ОНМЦК.docx")
Obrasheniye_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\Данные для тестирования 01.06.26\0. Обращение о проведении закупки.docx")

# contract_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\doci_primery\PACK_06_05\4_Проект_контракта_шины_и_комплектующие.docx")
# plan_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\doci_primery\PACK_06_05\1_Заявка_на_включение_в_план_график.docx")
# zapiska_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\PACK_06_05\5. Пояснительная записка (1).docx")
# OOZ_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\PACK_06_05\3. ООЗ автошины и комплектующие.docx")
# ONMCK_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\PACK_06_05\2_ОЦК_метод_сопостовимых_рыночных_цен_без_учета_ПП_1875_.docx")
# Obrasheniye_path = Path(r"C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\PACK_06_05\0. Обращение ПД (1).docx")

In [22]:
table_ktry_names, table_characteristics, ktry_codes = _parse_contract_characteristics(contract_path)

['58.29.11.000-00000003', '58.29.11.000-00000003', '58.29.11.000-00000003', '58.29.11.000-00000003', '58.29.11.000-00000003', '58.29.11.000-00000003', '58.29.11.000-00000003', '58.29.11.000-00000003']


In [23]:
ktry_codes

{'58.29.11.000-00000003'}

In [24]:
# print(table_ktry_names)
table_characteristics

{'58.29.11.000-00000003': {'Класс программ для электронных вычислительных машин и баз данных': '(12.10) Программное обеспечение для решения отраслевых задач в области информации и связи',
  'Способ предоставления': 'Копия электронного экземпляра',
  'Вид лицензии': 'Простая (неисключительная)',
  'Мониторинг точек беспроводного доступа': 'Наличие',
  'Обновление микропрограммного обеспечения точек беспроводного доступа': 'Наличие',
  'Управление и настройка точек беспроводного доступа': 'Наличие',
  'Срок действия лицензии': 'Бессрочно',
  'Товарный знак': 'ELTEX*'}}

In [44]:
%load_ext autoreload
%autoreload 2

from govno_model.check_registry import get_regestry_response_okpd_ktry, compare_characteristics
from services.procurement_reference_registry import ProcurementReferenceRegistry

def filter_plan_points(plan_points: List[str], keywords: List[str]) -> List[str]:
    plan_points_use = [
        plan_point
        for plan_point in plan_points
        if any(keyword.lower() in plan_point.lower() for keyword in keywords)
    ]
    return plan_points_use
REGISTRY_DIR = r"data\parsed_tables"
registry = ProcurementReferenceRegistry(REGISTRY_DIR)
# registry.get_ktru_common_info("26.30.23.000-00000016")
# registry.get_ktru_characteristics(ktry_codes[0])
plan_points = _parse_plan_points(plan_path)

procurement_method = filter_plan_points(
    plan_points,
    ["Способ выбора поставщика", "Способ выбора поставщика/исполнителя"],
)
okdp_plan = filter_plan_points(
    plan_points,
    ["ОКПД"],
)[0].split(":")[1].split("-")[0].strip()

characteristics_compare_result = compare_characteristics(
    OOZ_path,
    procurement_method,
    okdp_plan,
    REGISTRY_DIR,
)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
['58.29.11.0', '58.29.11.00', '58.29.11.000']
['58.29.11.0', '58.29.11.00', '58.29.11.000', '58.29.1', '58.29.11']
['58.29.11.0', '58.29.11.00', '58.29.11.000', '58.29.1', '58.29.11', '58.2', '58.29']


In [45]:
characteristics_compare_result

{'58.29.11.000-00000003': {'procurement_method': 'Способ выбора поставщика/исполнителя: Электронный аукцион',
  'selected_okpd2': '58.29.11.000',
  'matched_okpd2': '58.29',
  'okpd2_candidates': ['58.29.11.000'],
  'has_ktru_characteristics': True,
  'pp1875_appendix': 'appendix_1',
  'pp1875_position_number': 146,
  'can_add_extra_characteristics': True,
  'reason': 'Связанный ОКПД2 не попадает в специальные позиции ПП №1875. Дополнительные характеристики допустимы.',
  'field_errors': {}}}

In [46]:
import numpy as np
result = _parse_onmck_pricies(ONMCK_path)

print(result)

№1 Программное обеспечение,; КТРУ 58.29.11.000-00000003 | <ok>коэффициент вариации:  3.47%</ok> | Цены: [4461.0, 4377.0, 4169.0]
№ ИТОГО                                                 | <ok>коэффициент вариации:  3.47%</ok> | Цены: [2257266.0, 2214762.0, 2109514.0]


In [47]:
contract_points = _parse_contract_points(contract_path)
print(contract_points)

Наименование,; ОКПД2/КТРУ
Программное обеспечение,; КТРУ 58.29.11.000-00000003


In [78]:
ooz_points = _parse_ooz_points(OOZ_path)
print(ooz_points)

Наименование,; ОКПД2/КТРУ
Программное обеспечение,; КТРУ 58.29.11.000-00000003
| Наименование, ; ОКПД2/КТРУ: Программное обеспечение,; КТРУ 58.29.11.000-00000003 | Количество, штук: 506 |


In [56]:
smart_keywords = [
    "Код ОКПД",
    "Код позиции КТРУ",
    "Количество",
]
plan_points = _parse_plan_points(plan_path)
plan_points_use = filter_plan_points(plan_points, smart_keywords)
parse_ktry_entries(plan_points_use[1])

[{'ktru_code': '58.29.11.000-00000003',
  'name': 'У данного КТРУ не указано имя'}]

In [66]:
contract_points = _parse_contract_points(contract_path)
parser_contract = DocumentParser(contract_path)
parser_contract.extract_tables_columns(keywords=["Наименование,", "Количество"])

'| Наименование, ; ОКПД2/КТРУ: Программное обеспечение,; КТРУ 58.29.11.000-00000003 | Количество, штук: 506 |'

In [67]:
contract_points

'Наименование,; ОКПД2/КТРУ\nПрограммное обеспечение,; КТРУ 58.29.11.000-00000003\n| Наименование, ; ОКПД2/КТРУ: Программное обеспечение,; КТРУ 58.29.11.000-00000003 | Количество, штук: 506 |'

In [50]:
from govno_model.ai_service import get_ai_service
from docx import Document

ai_service = get_ai_service()
result = ai_service.process_query(
    plan_path = plan_path,
    contract_path = contract_path,
    ooz_path = OOZ_path,
    zapiska_path = zapiska_path,
    ONMCK_path = ONMCK_path,
    Obrasheniye_path = Obrasheniye_path,
)

ai_response = result["ai_response"]

output_path = project_root / "govno_model" / "govno_analysis.docx"

doc = Document()
doc.add_paragraph(ai_response)
doc.save(output_path)

print("Saved to:", output_path)
print()
print(ai_response[:3000])

Ошибка при парсинге КТРУ plan_points_use[1]: not enough values to unpack (expected 2, got 1)
['58.29.50.0', '58.29.50.00', '58.29.50.000']
['58.29.50.0', '58.29.50.00', '58.29.50.000', '58.29.5', '58.29.50']
['58.29.50.0', '58.29.50.00', '58.29.50.000', '58.29.5', '58.29.50', '58.2', '58.29']
Saved to: c:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\govno_model\govno_analysis.docx

<b>1) Проверка КТРУ через сервис zakupki.gov.ru:</b>

Не удалось распарсить КТРУ в Плане-графике


<b>2) Проверка ОКПД на вхождение в постановление 1875:</b>

<warn><ins>Обратите внимание</ins> на Код 58.29.50.000.</warn>
Родительский код 58.29 <ins>Входит в перечень</ins> "ПРИЛОЖЕНИЕ N 1 к постановлению Правительства Российской Федерации от 23 декабря 2024 г. N 1875 — Перечень товаров (в том числе поставляемых при выполнении закупаемых работ, оказании закупаемых услу...".
Эталонное наименование: Программа для электронной вычислительной машины и (или) базы данных.
Провер

# ПРОЕКТ КОНТРАКТА

In [ ]:
contract_path = Path("C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\\6_Проект_контракта_поставка_мебели_в2_2.docx")
plan_path = Path("C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\\5_Заявка_на_включение_в_план_график_1.docx")

parser_contract = DocumentParser(contract_path)
paragraphs_contract = parser_contract.extract_clean_text()
tables_contract = parser_contract.table_to_markdown()
contract_full_text = ("Название: " + paragraphs_contract + "\n\n" + tables_contract).strip()

KTRY_OKPD = parser_contract.extract_table_cells_by_keyword(["ОКПД", "КТРУ"])
KTRY_OKPD

import re

clean_items = []
for key in KTRY_OKPD:
    for item in KTRY_OKPD[key]:
        item = item.strip()
        item = re.sub(r"\s*;\s*", "; ", item)
        item = re.sub(r"(;\s*){2,}", "; ", item)
        item = item.rstrip("; ").strip()
        clean_items.append(item)

KTRY_OKPD_str = "\n".join(dict.fromkeys(clean_items))
print(KTRY_OKPD_str)



# ПЛАН ГРАФИК

In [21]:
parser_plan = PlanParser(plan_path)
plan_points = parser_plan.extract_table_kv_from_docx()
key_words = [
    "ОКПД",
    "КТРУ",
    "Количество"
    ]
plan_points_use = [
    plan_point
    for plan_point in plan_points
    if any(kw.lower() in plan_point.lower() for kw in key_words)
]
plan_points_str = "\n".join(plan_points_use)
print(plan_points_str)

Код ОКПД 2 и его наименования (расшифровки): 31.01.12.190 - Мебель офисная деревянная прочая; 31.01.11.150 - Мебель для сидения, преимущественно с металлическим каркасом; 31.01.12.110 - Столы письменные деревянные для офисов, административных помещений; 31.01.12.131 - Шкафы для одежды деревянные; 31.01.12.139 - Шкафы деревянные прочие; 31.01.12.150 - Тумбы офисные деревянные
Код позиции КТРУ: 31.01.11.150-00000003 - Стул на металлическом каркасе; 31.01.10.000-00000004 - Стол письменный; 31.01.12.131-00000001 - Шкаф для одежды деревянный; 31.01.12.139-00000001 - Шкаф деревянный для документов; 31.01.12.150-00000002 - Тумба офисная деревянная; 31.01.12.150-00000003 - Тумба офисная деревянная; 31.01.12.160-00000005 - Кресло офисное
Количество: Стол письменный - 12 шт.; Стол письменный - 1 шт.; Надставка настольная - 7 шт.; Тумба офисная деревянная - 12шт.; Тумба офисная деревянная - 4 шт.; Тумба офисная деревянная - 3 шт.; Подставка под системный блок - 14 шт.; Кресло офисное - 7 шт.; Шка

# ООЗ

In [ ]:
OOZ_path = Path("C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\\4. ООЗ_поставка_мебели (1).docx")

parser_ooz = DocumentParser(OOZ_path)
tables_ooz = parser_ooz.extract_table_cells_by_keyword(["ОКПД", "КТРУ"])
tables_ooz

clean_items = []
for key in tables_ooz:
    for item in tables_ooz[key]:
        item = item.strip()
        item = re.sub(r"\s*;\s*", "; ", item)
        item = re.sub(r"(;\s*){2,}", "; ", item)
        item = item.rstrip("; ").strip()
        clean_items.append(item)

tables_ooz_str = "\n".join(dict.fromkeys(clean_items))
print(tables_ooz_str)


# ПОЯСНИТЕЛЬНАЯ ЗАПИСКА

In [24]:
zapiska_path = Path("C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\\7. Пояснительная записка (1).docx")
parser_zapiska = DocumentParser(zapiska_path)

paragraphs_zapiska = parser_zapiska.extract_clean_text()
tables_zapiska = parser_zapiska.table_to_markdown()
zapiska_full_text = ("Название: " + paragraphs_zapiska + "\n\n" + tables_zapiska).strip()

print(zapiska_full_text)

Название: ПОЯСНИТЕЛЬНАЯ ЗАПИСКА

Электронный аукцион на поставку офисной мебели.

Состав поставки:

Стол письменный - 12 шт.

Стол письменный - 1 шт.

Надставка настольная - 7 шт.

Тумба офисная деревянная - 12шт.

Тумба офисная деревянная - 4 шт.

Тумба офисная деревянная - 3 шт.

Подставка под системный блок - 14 шт.

Кресло офисное - 7 шт.

Шкаф для одежды деревянный - 4 шт.

Шкаф для одежды деревянный - 2 шт.

Шкаф для одежды деревянный - 1 шт.

Шкаф деревянный для документов - 1 шт.

Шкаф деревянный для документов - 1 шт.

Шкаф деревянный для документов - 2 шт.

Стул на металлическом каркасе - 4 шт.

Срок поставки: в течение 45 (сорока пяти) календарных дней с даты заключения Контракта.

ОНМЦК: 548 547 рублей 00 копеек

Цель: для обеспечения нужд отдела технической поддержки государственного бюджетного учреждения Новосибирской области "Центр информационных технологий Новосибирской области".

Начальник отдела В.В. Шамаев


# ОНМЦК количество

In [61]:
ONMCK_path = Path("C:\\Users\\egorg\\Documents\\RAG_минцифры\\git_repo_clone\\Documents-verification-Mintsifry\\doci_primery\\3. ОНМЦК_поставка мебели (1).docx")
parser_ONMCK = DocumentParser(ONMCK_path)
table_ONMCK = parser_ONMCK.extract_rows_region(keyword = "шт.")
print(table_ONMCK)
print(parser_ONMCK.extract_clean_text())



| Стол письменный | шт. | 1 |
| Надставка настольная | шт. | 7 |
| Стол письменный | шт. | 12 |
| Тумба офисная деревянная | шт. | 12 |
| Тумба офисная деревянная | шт. | 4 |
| Подставка под системный блок | шт. | 14 |
| Кресло офисное | шт. | 7 |
| Шкаф для одежды деревянный | шт. | 4 |
| Шкаф для одежды деревянный | шт. | 2 |
| Шкаф деревянный для документов | шт. | 1 |
| Шкаф деревянный для документов | шт. | 1 |
| Шкаф деревянный для документов | шт. | 2 |
| Тумба офисная деревянная | шт. | 3 |
| Стул на металлическом каркасе | шт. | 4 |
| Шкаф для одежды деревянный | шт. | 1 |
| Итого: | шт. | 75 |
ОБОСНОВАНИЕ НАЧАЛЬНОЙ (МАКСИМАЛЬНОЙ) ЦЕНЫ КОНТРАКТА

В соответствии со ст. 22 Федерального закона от 05.04.2013 № 44-ФЗ "О контрактной системе в сфере закупок товаров, работ, услуг для обеспечения государственных и муниципальных нужд", методическими рекомендациями по применению методов определения начальной (максимальной) цены контракта, цены контракта, заключаемого с единственным поста

# ТЕСТОВЫЙ ЗАПУСК МОДЕЛИ ДЛЯ СРАВНЕНИЯ КТРУ ОКПД И КОЛИЧЕСВА

In [9]:
from new_model.retriever import Retriever
plan_point = "Сроки поставки товара, выполнения работ," \
" оказания услуг по контракту: в течение 45 (сорока пяти)" \
" календарных дней с даты заключения контракта"

parser_onmck = DocumentParser(ONMCK_path)
parser_ONMCK = DocumentParser(ONMCK_path)
parser_zapiska = DocumentParser(zapiska_path)
parser_ooz = DocumentParser(OOZ_path)

paragraphs_zapiska = parser_zapiska.extract_clean_text()
tables_zapiska = parser_zapiska.table_to_markdown()

zapiska_full_text = ("Название: " + paragraphs_zapiska + "\n\n" + tables_zapiska).strip()
paragraphs_contract = parser_contract.extract_clean_text()
contract_full_text = paragraphs_contract.strip()
ooz_plain_text = parser_ooz.extract_clean_text()
onmck_plain_text = parser_onmck.extract_clean_text()

faiss = Retriever(embeddings=get_embeddings())
retriever = faiss.create_retriever(texts=[contract_full_text, zapiska_full_text, ooz_plain_text, onmck_plain_text], n=20)
docs = retriever.invoke(plan_point)
print("=" * 80)
print(f"plan_point {1}: {plan_point}")
print("-" * 80)

for j, doc in enumerate(docs, start=1):
    print(f"chunk {j}:")
    print(doc.page_content[:1500])
    print("-" * 80)

plan_point 1: Сроки поставки товара, выполнения работ, оказания услуг по контракту: в течение 45 (сорока пяти) календарных дней с даты заключения контракта
--------------------------------------------------------------------------------
chunk 1:
системы в сфере закупок (далее - место доставки), в срок в течение 45 (сорока пяти) календарных дней с даты заключения Контракта. Дата заключения Контракта не входит в срок поставки Товара
--------------------------------------------------------------------------------
chunk 2:
Таблица 1 Место поставки Товаров

Срок поставки: в течение 45 календарных дней с даты заключения контракта.

Характеристики поставляемого Товара:

Таблица №2 Функциональные, технические и качественные характеристики Товара

Условия поставки Товара:
--------------------------------------------------------------------------------
chunk 3:
Таблица №1 Место поставки Товаров

Срок поставки: в течение 45 (сорока пяти) календарных дней с даты заключения контракта.

Характеристи

In [14]:
from new_model.retriever import BM25TextRetriever

bm25 = BM25TextRetriever()
retriever = bm25.create_retriever(
    texts=[contract_full_text, zapiska_full_text, ooz_plain_text, onmck_plain_text],
    n=7,
)

docs = retriever.invoke(plan_point)

print("=" * 80)
print(f"plan_point 1: {plan_point}")
print("-" * 80)

for j, doc in enumerate(docs, start=1):
    print(f"chunk {j}:")
    print(doc.page_content[:1500])
    print("-" * 80)


plan_point 1: Сроки поставки товара, выполнения работ, оказания услуг по контракту: в течение 45 (сорока пяти) календарных дней с даты заключения контракта
--------------------------------------------------------------------------------
chunk 1:
системы в сфере закупок (далее - место доставки), в срок в течение 45 (сорока пяти) календарных дней с даты заключения Контракта. Дата заключения Контракта не входит в срок поставки Товара
--------------------------------------------------------------------------------
chunk 2:
Таблица №1 Место поставки Товаров

Срок поставки: в течение 45 (сорока пяти) календарных дней с даты заключения контракта.

Характеристики поставляемого Товара:

Таблица №2 Функциональные, технические и качественные характеристики Товара

*Изображения носят описательный характер.
--------------------------------------------------------------------------------
chunk 3:
Срок поставки: в течение 45 (сорока пяти) календарных дней с даты заключения Контракта.

ОНМЦК: 548 547 

In [ ]:
# contract_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\doci_primery\ДЛЯ ГПТ\4_Контракт_на_поставку_моноблоков_1.docx")
# plan_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\doci_primery\ДЛЯ ГПТ\1. Заявка в ПГ (1).docx")
# zapiska_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\doci_primery\ДЛЯ ГПТ\5. Пояснительная записка (1).docx")
# OOZ_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\doci_primery\ДЛЯ ГПТ\3.ООЗ на поставку моноблоков (1).docx")
# ONMCK_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\doci_primery\ДЛЯ ГПТ\2. ОНМЦК (1).docx")
# Obrasheniye_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\doci_primery\ДЛЯ ГПТ\0_Обращение_о_проведении_закупки_1.docx")


In [16]:
output_path = project_root / "govno_model" / "govno_analysis.docx"
doc.save(output_path)

print("Saved to:", output_path)
print()
print(ai_response[:3000])

Saved to: c:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\govno_model\govno_analysis.docx


----------------------------------------------------------------------------------

----------------------------------------------------------------------------------
Код 26.20.15.140 напрямую не найден. Найден родительский код 26.20.15 в таблице 'ПРИЛОЖЕНИЕ N 2 к постановле...'. Эталонное наименование: Машины вычислительные электронные цифровые прочие, содержащие или не содержащие в одном корпусе одно или два из следующих устройств для автоматической обработки данных: запоминающие устройства, устройства ввода, устройства вывода. Проверьте соответствует ли ваше наименование 'Моноблоки'.
----------------------------------------------------------------------------------

----------------------------------------------------------------------------------
-------------------------
-------------------------

Проверка ОКПД:
Найдено в Плане-Графике:
26.20.15.140 - М

In [ ]:
# import sys
# from pathlib import Path
# from docx import Document

# project_root = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry")
# if str(project_root) not in sys.path:
#     sys.path.insert(0, str(project_root))

# from govno_model.ai_service import get_ai_service

# contract_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\doci_primery\6_Проект_контракта_поставка_мебели_в2_2.docx")
# plan_path = Path(r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry\doci_primery\5_Заявка_на_включение_в_план_график_1.docx")

# ai_service = get_ai_service()

# result = ai_service.process_query(Ц
#     doc1_path=str(plan_path),
#     doc2_path=str(contract_path),
# )

# ai_response = result["ai_response"]

# output_path = project_root / "govno_model" / "govno_analysis.docx"

# doc = Document()
# doc.add_paragraph(ai_response)
# doc.save(output_path)

# print("Saved to:", output_path)
# print()
# print(ai_response[:3000])
